### More than Chat: Fixing Multilingual Search Queries with LoRA finetuning on Apple Silicon

Code authored by: Sreejith Sreejayan <br>
Blog link: link <br>
<br>
Source: link

### imports

In [17]:
import subprocess
from mlx_lm import load, generate

### functions

In [18]:
def run_command_with_live_output(command: list[str]) -> None:
    """
    Courtesy of ChatGPT:
    Runs a command and prints its output line by line as it executes.

    Args:
        command (List[str]): The command and its arguments to be executed.

    Returns:
        None
    """
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)

    # Print the output line by line
    while True:
        output = process.stdout.readline()
        if output == '' and process.poll() is not None:
            break
        if output:
            print(output.strip())
        
    # Print the error output, if any
    err_output = process.stderr.read()
    if err_output:
        print(err_output)

In [19]:
def construct_shell_command(command: list[str]) -> str:
    
    return str(command).replace("'","").replace("[","").replace("]","").replace(",","")

In [20]:
# 1. The System Instruction (The "Brain")
# We define the persona, the specific linguistic capability (Manglish/Hinglish), 
# and the strict output format (JSON).
instruction_string = """You are a specialized Multilingual Query Corrector for an Indian quick-commerce application. \
You are an expert in interpreting "Manglish", "Tanglish", and phonetic typos in Indian contexts (e.g., typing 'dahi' for 'Curd' or 'kothimbir' for 'Coriander'). \

Your task is to take a noisy, misspelled, or vernacular user search query and map it to the correct Standard Product Name from the catalog. \
You must strictly output ONLY a valid JSON object in the following format: {"corrected": "Standard Product Name"}. \
Do not provide explanations or conversational text."""

# 2. The Prompt Builder (The "Syntax" for Gemma 3)
# Gemma uses <start_of_turn> tokens. We inject the instruction before the user input.
prompt_builder = lambda query: f"""<start_of_turn>user
{instruction_string}

User Query: {query}<end_of_turn>
<start_of_turn>model
"""

# Example Usage:
user_input = "nandni dahi packet"
print(prompt_builder(user_input))

<start_of_turn>user
You are a specialized Multilingual Query Corrector for an Indian quick-commerce application. You are an expert in interpreting "Manglish", "Tanglish", and phonetic typos in Indian contexts (e.g., typing 'dahi' for 'Curd' or 'kothimbir' for 'Coriander'). 
Your task is to take a noisy, misspelled, or vernacular user search query and map it to the correct Standard Product Name from the catalog. You must strictly output ONLY a valid JSON object in the following format: {"corrected": "Standard Product Name"}. Do not provide explanations or conversational text.

User Query: nandni dahi packet<end_of_turn>
<start_of_turn>model



### Quantize Model (optional)

In [21]:
hf_model_path = "<model_path>"

In [22]:
# define command to convert hf model to mlx format and save locally (-q flag quantizes model)
command = ['python', 'scripts/convert.py', '--hf-path', hf_model_path, '-q']

# print runable version of command (copy and paste into command line to run)
print("Use this command to convert the model:")
print(construct_shell_command(command))

Use this command to convert the model:
python scripts/convert.py --hf-path <model_path> -q


### Run inference with quantized model

In [23]:
model_path = "mlx-community/gemma-3-4b-it-qat-4bit"
prompt = prompt_builder("nandni dahi packet")
max_tokens = 140

In [24]:
model, tokenizer = load("mlx-community/gemma-3-4b-it-qat-4bit")
response = generate(model, tokenizer, prompt=prompt, max_tokens = max_tokens,verbose=True)

Fetching 12 files:   0%|          | 0/12 [00:00<?, ?it/s]

{"corrected": "Curd Packet"}

Prompt: 144 tokens, 190.257 tokens-per-sec
Generation: 10 tokens, 45.333 tokens-per-sec
Peak memory: 5.122 GB


### Fine-tune with LoRA

In [25]:
num_iters = "100"
steps_per_eval = "10"
val_batches = "-1" # use all
learning_rate = "1e-5" # same as default
num_layers = 16 # same as default
# no dropout or weight decay :(

In [26]:
# define command
command = ['python', 'scripts/lora.py', '--model', model_path, '--train', '--iters', num_iters, '--steps-per-eval', steps_per_eval, '--val-batches', val_batches, '--learning-rate', learning_rate, '--lora-layers', num_layers, '--test']

# run command and print results continuously (doesn't print loss during training)
# run_command_with_live_output(command)

In [27]:
# print command to run in command line directly
print("Use this command to run the training:")
print(construct_shell_command(command))

Use this command to run the training:
python scripts/lora.py --model mlx-community/gemma-3-4b-it-qat-4bit --train --iters 100 --steps-per-eval 10 --val-batches -1 --learning-rate 1e-5 --lora-layers 16 --test


### Run inference with fine-tuned model

In [28]:
adapter_path = "adapters.npz" # same as default
max_tokens_str = str(max_tokens)

In [29]:
# define command
command = ['python', 'scripts/lora.py', '--model', model_path, '--adapter-file', adapter_path, '--max-tokens', max_tokens_str, '--prompt', prompt]

# run command and print results continuously
print(construct_shell_command(command))
run_command_with_live_output(command)

python scripts/lora.py --model mlx-community/gemma-3-4b-it-qat-4bit --adapter-file adapters.npz --max-tokens 140 --prompt <start_of_turn>user\nYou are a specialized Multilingual Query Corrector for an Indian quick-commerce application. You are an expert in interpreting "Manglish" "Tanglish" and phonetic typos in Indian contexts (e.g. typing \dahi\ for \Curd\ or \kothimbir\ for \Coriander\). \nYour task is to take a noisy misspelled or vernacular user search query and map it to the correct Standard Product Name from the catalog. You must strictly output ONLY a valid JSON object in the following format: {"corrected": "Standard Product Name"}. Do not provide explanations or conversational text.\n\nUser Query: nandni dahi packet<end_of_turn>\n<start_of_turn>model\n


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Loading pretrained model
Model type: gemma3
Applying LoRA to last 16 of 34 layers
Total parameters 712.009M
Trainable parameters 0.524M
Loading datasets
Generating
<start_of_turn>user
You are a specialized Multilingual Query Corrector for an Indian quick-commerce application. You are an expert in interpreting "Manglish", "Tanglish", and phonetic typos in Indian contexts (e.g., typing 'dahi' for 'Curd' or 'kothimbir' for 'Coriander').
Your task is to take a noisy, misspelled, or vernacular user search query and map it to the correct Standard Product Name from the catalog. You must strictly output ONLY a valid JSON object in the following format: {"corrected": "Standard Product Name"}. Do not provide explanations or conversational text.

User Query: nandni dahi packet<end_of_turn>
<start_of_turn>model
{"corrected": "Aashiri Curd"}<end_of_turn>

Fetching 12 files: 100%|██████████| 12/12 [00:00<00:00, 179116.19it/s]



#### a harder query

In [30]:
query = "kothimbir bunch"
prompt = prompt_builder(query)

In [31]:
# define command
command = ['python', 'scripts/lora.py', '--model', model_path, '--adapter-file', adapter_path, '--max-tokens', max_tokens_str, '--prompt', prompt]

# run command and print results continuously
run_command_with_live_output(command)

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Loading pretrained model
Model type: gemma3
Applying LoRA to last 16 of 34 layers
Total parameters 712.009M
Trainable parameters 0.524M
Loading datasets
Generating
<start_of_turn>user
You are a specialized Multilingual Query Corrector for an Indian quick-commerce application. You are an expert in interpreting "Manglish", "Tanglish", and phonetic typos in Indian contexts (e.g., typing 'dahi' for 'Curd' or 'kothimbir' for 'Coriander').
Your task is to take a noisy, misspelled, or vernacular user search query and map it to the correct Standard Product Name from the catalog. You must strictly output ONLY a valid JSON object in the following format: {"corrected": "Standard Product Name"}. Do not provide explanations or conversational text.

User Query: kothimbir bunch<end_of_turn>
<start_of_turn>model
{"corrected": "Coriander"}<end_of_turn>

Fetching 12 files: 100%|██████████| 12/12 [00:00<00:00, 25053.09it/s]

